# HPT using Optuna

Objective Function:
- Search Space
- Model init
- param init
- training loop
- evaluation loop

In [1]:
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.optim as optim

In [2]:
torch.manual_seed(42)

In [3]:
device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: mps


In [4]:
df = pd.read_csv('data/fashion-mnist_train.csv')
df.head(2)

,label,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,pixel9,...,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783,pixel784
0,2,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,9,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [5]:
X = df.iloc[:,1:].values
y = df.iloc[:,0].values

In [6]:
X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=0.2, random_state=42)

In [7]:
X_train = X_train/255.0
X_test = X_test/255.0

In [8]:
#defining the dataset
class CustomDataset(Dataset):
    def __init__(self, features, label):
        self.features = torch.tensor(features, dtype=torch.float32)
        self.label = torch.tensor(label, dtype=torch.long)
    
    def __len__(self):
        return self.features.shape[0]
    
    def __getitem__(self,index):
        return self.features[index], self.label[index]

In [9]:
train_dataset = CustomDataset(X_train, y_train)
#create test dataset
test_dataset = CustomDataset(X_test,y_test)

In [10]:
#create train and test loader

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False, pin_memory=True)

In [19]:
class Mynn(nn.Module):
    def __init__(self, input_dim, output_dim, num_hidden_layers, neurons_per_layer):
        super().__init__()
        layers = []
        for i in range(num_hidden_layers):
            layers.append(nn.Linear(input_dim, neurons_per_layer))
            layers.append(nn.BatchNorm1d(neurons_per_layer))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(0.3))
            input_dim = neurons_per_layer
        layers.append(nn.Linear(neurons_per_layer, output_dim))

        self.model = nn.Sequential(*layers)

    def forward(self,x):
        return self.model(x)

In [20]:
#objective function

import optuna
def objective(trial):

    #search space
    num_hidden_layers = trial.suggest_int("num_hidden_layers", 1,5)
    neurons_per_layer = trial.suggest_int("neurons_per_layer", 8, 128, step=8)

    #model init
    input_dim = 784
    output_dim = 10
    model = Mynn(input_dim, output_dim, num_hidden_layers, neurons_per_layer)
    model.to(device)

    #param init
    epochs = 50
    learning_rate = 0.1

    #optimizer selection
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.SGD(model.parameters(), learning_rate, weight_decay=1e-4)

    #training loop

    for epoch in range(epochs):
        for batch_features, batch_label in train_loader:
            batch_features, batch_label = batch_features.to(device), batch_label.to(device)
            #forwardpass
            outputs = model(batch_features)
            #calculate loss
            loss = criterion(outputs, batch_label)
            #backpass
            optimizer.zero_grad()
            loss.backward()
            #updateparams
            optimizer.step()
    
    #evaluation
    model.eval()

    total = 0
    correct = 0

    with torch.no_grad():
        for batch_features, batch_label in test_loader:
            batch_features, batch_label = batch_features.to(device), batch_label.to(device)
            output = model(batch_features)
            _, predicted = torch.max(output,1)
            
            total = total + batch_label.shape[0]
            correct = correct + (predicted == batch_label).sum().item()
        
        accuracy = (correct/total) 
    return accuracy


In [21]:
study = optuna.create_study(direction="maximize")

[I 2026-01-23 12:03:54,918] A new study created in memory with name: no-name-05c12469-609f-47cd-8955-5d5d5256cc25


In [22]:
study.optimize(objective,n_trials=10)

/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)
[I 2026-01-23 12:04:46,805] Trial 0 finished with value: 0.88275 and parameters: {'num_hidden_layers': 1, 'neurons_per_layer': 104}. Best is trial 0 with value: 0.88275.
[I 2026-01-23 12:05:44,303] Trial 1 finished with value: 0.8625833333333334 and parameters: {'num_hidden_layers': 1, 'neurons_per_layer': 24}. Best is trial 0 with value: 0.88275.
[I 2026-01-23 12:06:38,496] Trial 2 finished with value: 0.8831666666666667 and parameters: {'num_hidden_layers': 2, 'neurons_per_layer': 104}. Best is trial 2 with value: 0.8831666666666667.
[I 2026-01-23 12:07:50,897] Trial 3 finished with value: 0.8563333333333333 and parameters: {'num_hidden_layers': 3, 'neurons_per_layer': 24}. Best is trial 2 with value: 0.8831666666666667.
[I 2026-0

# Advanced with All Parameters Hypertuned

In [32]:
df = pd.read_csv('data/fashion-mnist_train.csv')

In [33]:
X = df.iloc[:,1:].values
y = df.iloc[:,0].values

X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=0.2, random_state=42)

X_train = X_train/255.0
X_test = X_test/255.0

#defining the dataset
class CustomDataset(Dataset):
    def __init__(self, features, label):
        self.features = torch.tensor(features, dtype=torch.float32)
        self.label = torch.tensor(label, dtype=torch.long)
    
    def __len__(self):
        return self.features.shape[0]
    
    def __getitem__(self,index):
        return self.features[index], self.label[index]

In [34]:
train_dataset = CustomDataset(X_train, y_train)
#create test dataset
test_dataset = CustomDataset(X_test,y_test)

In [35]:
class Mynn(nn.Module):
    def __init__(self, input_dim, output_dim, num_hidden_layers, neurons_per_layer, dropout_rate):
        super().__init__()
        layers = []
        for i in range(num_hidden_layers):
            layers.append(nn.Linear(input_dim, neurons_per_layer))
            layers.append(nn.BatchNorm1d(neurons_per_layer))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout_rate))
            input_dim = neurons_per_layer
        layers.append(nn.Linear(neurons_per_layer, output_dim))

        self.model = nn.Sequential(*layers)

    def forward(self,x):
        return self.model(x)

In [36]:
#objective function

import optuna
def objective(trial):

    #search space
    num_hidden_layers = trial.suggest_int("num_hidden_layers", 1,5)
    neurons_per_layer = trial.suggest_int("neurons_per_layer", 8, 128, step=8)
    epochs = trial.suggest_int("epochs", 10,50, step=10)
    learning_rate = trial.suggest_float("learning_rate", 1e-5, 1e-1, log=True)
    dropout_rate = trial.suggest_float("dropout_rate", 0.1, 0.5, step=1)
    batch_size = trial.suggest_categorical("batch_size", [16,32,64,128])
    optimizer_name = trial.suggest_categorical('optimizer', ['Adam', 'SGD', 'RMSprop'])
    weight_decay = trial.suggest_float("weight_decay", 1e-5, 1e-2, log=True)

    #create train and test loader
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, pin_memory=True)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, pin_memory=True)

    #model init
    input_dim = 784
    output_dim = 10
    model = Mynn(input_dim, output_dim, num_hidden_layers, neurons_per_layer, dropout_rate)
    model.to(device)

    #optimizer selection
    criterion = nn.CrossEntropyLoss()
    

    if optimizer_name == 'Adam':
        optimizer = optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
    elif optimizer_name == 'SGD':
        optimizer = optim.SGD(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
    else:
        optimizer = optim.RMSprop(model.parameters(), lr=learning_rate, weight_decay=weight_decay )

    #training loop

    for epoch in range(epochs):
        for batch_features, batch_label in train_loader:
            batch_features, batch_label = batch_features.to(device), batch_label.to(device)
            #forwardpass
            outputs = model(batch_features)
            #calculate loss
            loss = criterion(outputs, batch_label)
            #backpass
            optimizer.zero_grad()
            loss.backward()
            #updateparams
            optimizer.step()
    
    #evaluation
    model.eval()

    total = 0
    correct = 0

    with torch.no_grad():
        for batch_features, batch_label in test_loader:
            batch_features, batch_label = batch_features.to(device), batch_label.to(device)
            output = model(batch_features)
            _, predicted = torch.max(output,1)
            
            total = total + batch_label.shape[0]
            correct = correct + (predicted == batch_label).sum().item()
        
        accuracy = (correct/total) 
    return accuracy
 

In [37]:
study = optuna.create_study(direction="maximize")

[I 2026-01-23 13:10:41,787] A new study created in memory with name: no-name-593d1d55-bd55-4cca-9cf8-bf3b521c82f6


In [38]:
study.optimize(objective,n_trials=3)

/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/optuna/distributions.py:671: UserWarning: The distribution is specified by [0.1, 0.5] and step=1, but the range is not divisible by `step`. It will be replaced with [0.1, 0.1].
  optuna_warn(
/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)
[I 2026-01-23 13:12:01,529] Trial 0 finished with value: 0.8864166666666666 and parameters: {'num_hidden_layers': 1, 'neurons_per_layer': 72, 'epochs': 30, 'learning_rate': 0.00023862651492628195, 'dropout_rate': 0.1, 'batch_size': 64, 'optimizer': 'Adam', 'weight_decay': 1.7673389169778277e-05}. Best is trial 0 with value: 0.8864166666666666.
[I 2026-01-23 13:17:44,151] Trial 1 finished with value: 0.8208333333333333 and parameters: {'num_hidden_layers': 3, 'neuron